# ML-10 — Content Action Playbook

This playbook turns descriptive clusters into an editor-review queue. Recommendations and scores are decision-support aids, not automated actions or outcome predictions.

## 1. Fit the descriptive clustering model

K-Means returns arbitrary numeric IDs. The archetype labels below are assigned after inspecting the feature distributions, so they are human interpretations rather than direct model outputs.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

possible_paths = [
    '../../data/raw/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv',
    '/content/content_refresh_anonymized.csv',
]
data_path = next((path for path in possible_paths if os.path.exists(path)), None)
if data_path is None:
    raise FileNotFoundError('Could not locate content_refresh_anonymized.csv')

df_raw = pd.read_csv(data_path)
df_clean = df_raw.loc[
    (df_raw['impressions_90d'] >= 10)
    & (df_raw['content_age_days'] >= 90)
    & (df_raw['avg_position'] > 0)
] .copy()

X = pd.DataFrame({
    'impressions_log': np.log1p(df_clean['impressions_90d']),
    'avg_position': df_clean['avg_position'],
    'ctr': df_clean['ctr'],
    'staleness_log': np.log1p(df_clean['days_since_last_update']),
    'engagement_rate': df_clean['engagement_rate'],
})
X_scaled = StandardScaler().fit_transform(X)
df_clean['cluster'] = KMeans(n_clusters=5, random_state=42, n_init=20).fit_predict(X_scaled)

cluster_names = {
    0: 'Stale, High-Reach',
    1: 'Current, High-Reach',
    2: 'High-Engagement',
    3: 'Low-Visibility',
    4: 'High-CTR, Low-Reach',
}
df_clean['archetype'] = df_clean['cluster'].map(cluster_names)

profile = df_clean.groupby('archetype').agg(
    pages=('content_id', 'size'),
    median_impressions=('impressions_90d', 'median'),
    median_position=('avg_position', 'median'),
    median_ctr=('ctr', 'median'),
    median_staleness=('days_since_last_update', 'median'),
    median_engagement=('engagement_rate', 'median'),
).round(2)
display(profile)


## 2. Transparent action heuristic

The priority score is a manually specified decision heuristic, not a supervised model trained on downstream outcomes. It combines reach (40%), ranking opportunity (35%), and staleness (25%). Higher average position means lower search placement, so the ranking component represents opportunity for review rather than quality.

In [ ]:
action_map = {
    'Stale, High-Reach': 'Review for factual and content refresh',
    'Current, High-Reach': 'Monitor and protect',
    'High-Engagement': 'Review for scalable experience patterns',
    'Low-Visibility': 'Review intent, internal links, or consolidation',
    'High-CTR, Low-Reach': 'Investigate query coverage and internal linking',
}
reason_map = {
    'Stale, High-Reach': 'STALE_HIGH_REACH',
    'Current, High-Reach': 'CURRENT_HIGH_REACH',
    'High-Engagement': 'HIGH_ENGAGEMENT',
    'Low-Visibility': 'LOW_VISIBILITY',
    'High-CTR, Low-Reach': 'HIGH_CTR_LOW_REACH',
}
df_clean['recommended_action'] = df_clean['archetype'].map(action_map)
df_clean['reason_code'] = df_clean['archetype'].map(reason_map)

position_p95 = df_clean['avg_position'].quantile(0.95)
def add_priority_score(frame, reach_weight, ranking_weight, staleness_weight):
    scored = frame.copy()
    reach = np.log1p(scored['impressions_90d']) / np.log1p(scored['impressions_90d'].max())
    ranking_opportunity = np.clip((scored['avg_position'] - 1) / (position_p95 - 1), 0, 1)
    staleness = np.clip(scored['days_since_last_update'] / 365, 0, 1)
    scored['action_priority_score'] = (
        reach_weight * reach
        + ranking_weight * ranking_opportunity
        + staleness_weight * staleness
    )
    return scored

df_ranked = add_priority_score(df_clean, 0.40, 0.35, 0.25)
df_ranked = df_ranked.sort_values('action_priority_score', ascending=False).reset_index(drop=True)
df_ranked['rank'] = df_ranked.index + 1

preview_columns = [
    'rank', 'archetype', 'recommended_action', 'reason_code',
    'action_priority_score', 'impressions_90d', 'avg_position',
    'days_since_last_update',
]
display(df_ranked[preview_columns].head(20).round({'action_priority_score': 3}))


## 3. Sensitivity analysis

The alternatives below vary the manual weights. If the same pages remain near the top, the review queue is less sensitive to this reasonable weighting choice; this does not make the score causal or learned.

In [ ]:
weight_sets = {
    '40 / 35 / 25 (reference)': (0.40, 0.35, 0.25),
    '50 / 30 / 20': (0.50, 0.30, 0.20),
    '30 / 40 / 30': (0.30, 0.40, 0.30),
}
reference_top20 = set(df_ranked.head(20)['content_id'])
sensitivity = []
for name, weights in weight_sets.items():
    scenario = add_priority_score(df_clean, *weights).sort_values(
        'action_priority_score', ascending=False
    ).head(20)
    scenario_top20 = set(scenario['content_id'])
    sensitivity.append({
        'weights (reach / ranking / staleness)': name,
        'top_20_overlap_with_reference': len(reference_top20 & scenario_top20),
        'top_20_jaccard_with_reference': len(reference_top20 & scenario_top20) / len(reference_top20 | scenario_top20),
    })

display(pd.DataFrame(sensitivity).round(3))


## 4. Human review, limits, and no-go list

Before acting, a reviewer must check SERP query intent, branded versus non-branded demand, redirects/migrations, backlinks, factual accuracy, and topic risk. Never automatically delete or redirect a page, and never auto-publish AI text on medical, legal, or financial pages. Pages under 90 days old, pages with fewer than 10 impressions, and pages without a valid average position remain outside this playbook.

In [ ]:
governance = pd.DataFrame({
    'recommendation': list(action_map.values()),
    'minimum human check': [
        'Verify factual accuracy, intent, and current information',
        'Check ranking volatility and technical changes',
        'Confirm a transferable pattern rather than a one-off audience effect',
        'Review backlinks, conversion role, and consolidation risk',
        'Inspect query intent, page coverage, and on-page relevance',
    ],
    'automation_level': ['Decision-support only'] * 5,
})
display(governance)


## 5. Export and monitoring

The generated queue is an auditable snapshot. Re-run the pipeline when the rolling observation window changes, when cluster coherence degrades, when centroids materially drift, or after a confirmed search-environment change.

In [ ]:
output_dir = '../../outputs' if os.path.isdir('../../outputs') else 'outputs'
os.makedirs(output_dir, exist_ok=True)
export_columns = [
    'rank', 'content_id', 'client_id', 'archetype', 'recommended_action',
    'reason_code', 'action_priority_score', 'impressions_90d', 'avg_position',
    'ctr', 'days_since_last_update', 'engagement_rate',
]
queue_path = os.path.join(output_dir, 'content_action_playbook_queue.csv')
df_ranked[export_columns].to_csv(queue_path, index=False)
print(f'Exported decision-support queue: {queue_path}')


## Self-check

- [x] The score is documented as a heuristic with explicit weights.
- [x] Alternative weights are compared.
- [x] Human review is mandatory before action.
- [x] Archetype labels are distinguished from K-Means IDs.